# Lab 28 — vLLM Serving trên Kaggle GPU (T4)

Notebook này phục vụ cho **IP07 (Model → vLLM / SGLang serving)** trong bài **Lab 28 Track 2**.

### ⚠️ Cài đặt bắt buộc trên Kaggle trước khi chạy:
1. Ở thanh menu bên phải (Notebook options):
   - **Accelerator**: Chọn `GPU T4 x 1` (hoặc `GPU T4 x 2`).
   - **Internet**: Phải gạt sang **ON** (để tải model và Cloudflared tunnel).
2. Sau khi chạy xong Cell 3 & 4, notebook sẽ in ra **Public Tunnel URL**.
3. Anh chỉ cần copy URL đó về máy local để chạy kiểm thử.

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi

## 2. Cài đặt vLLM và Cloudflared Tunnel

In [ ]:
# Gỡ torchaudio bị lệch phiên bản CUDA trên Colab
!pip uninstall -y torchaudio

# Cài đặt vLLM v0.28.0 (khớp chuẩn bài lab)
!pip install -q "vllm==0.28.0" httpx requests

# Tải Cloudflared để mở public URL an toàn, miễn phí, không cần đăng ký tài khoản
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
print("✅ Đã tải và cấp quyền thực thi cho cloudflared!")

## 3. Khởi động Cloudflared Tunnel (Public URL)

In [ ]:
import subprocess
import time
import re

# Chạy cloudflared trỏ về port 8000 nội bộ
tunnel_proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

tunnel_url = None
print("⏳ Đang khởi tạo Cloudflare tunnel...")
for _ in range(40):
    line = tunnel_proc.stdout.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        break
    time.sleep(0.5)

if tunnel_url:
    print("\n" + "="*65)
    print(f"🎉 TUNNEL URL CỦA ANH:")
    print(f"   {tunnel_url}")
    print(f"\n👉 LAB28_VLLM_BASE_URL (dùng trên máy local):")
    print(f"   {tunnel_url}/v1")
    print("="*65 + "\n")
else:
    print("❌ Chưa tìm thấy URL, hãy chạy lại cell này hoặc xem output của cloudflared.")

## 4. Khởi động vLLM Server (Chạy nền với model Qwen/Qwen3-1.7B)

In [ ]:
import subprocess
import time
import urllib.request

MODEL_ID = "Qwen/Qwen3-1.7B"

print(f"⏳ Đang khởi động vLLM với model {MODEL_ID}...")
vllm_log = open("vllm.log", "w")
vllm_proc = subprocess.Popen(
    [
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", MODEL_ID,
        "--host", "0.0.0.0",
        "--port", "8000",
        "--dtype", "half",
        "--max-model-len", "4096",
        "--gpu-memory-utilization", "0.85"
    ],
    stdout=vllm_log,
    stderr=subprocess.STDOUT
)

# Chờ đến khi vLLM sẵn sàng (/health trả về 200)
ready = False
for i in range(120):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=2) as resp:
            if resp.status == 200:
                ready = True
                break
    except Exception:
        pass
    time.sleep(2)
    if i % 10 == 0 and i > 0:
        print(f"   ...đang nạp model và khởi tạo KV cache ({i*2}s)...")

if ready:
    print("\n🚀 vLLM SERVER ĐÃ SẴN SÀNG!")
    print(f"   Model: {MODEL_ID}")
    print(f"   Public Endpoint: {tunnel_url}/v1")
else:
    print("❌ Timeout khi chờ vLLM. Log chi tiết 30 dòng cuối:")
    with open("vllm.log", "r") as f:
        print("".join(f.readlines()[-30:]))

## 5. Kiểm tra Health & Identity (Chuẩn IP07)

In [ ]:
import requests

base_url = "http://127.0.0.1:8000"

# 1. Kiểm tra /version
print("1. /version:", requests.get(f"{base_url}/version").json())

# 2. Kiểm tra /v1/models
print("2. /v1/models:", requests.get(f"{base_url}/v1/models").json())

# 3. Kiểm tra metrics vllm:
metrics = requests.get(f"{base_url}/metrics").text
vllm_lines = [l for l in metrics.splitlines() if l.startswith("vllm:")][:5]
print("3. Metrics có tiền tố 'vllm:':", vllm_lines)

print("\n✅ Đạt 100% điều kiện Identity của IP07!")

## 6. Hướng dẫn lệnh chạy trên máy local của anh

Sau khi có `tunnel_url` (ví dụ: `https://abc-xyz.trycloudflare.com`), anh mở PowerShell trên máy local và chạy:

```powershell
# 1. Gán URL tunnel vào biến môi trường
$env:LAB28_VLLM_BASE_URL="https://abc-xyz.trycloudflare.com/v1"
$env:LAB28_VLLM_REQUIRE_REAL="true"

# 2. Kiểm tra lại readiness (sẽ thấy IP07 chuyển sang ready)
uv run lab28 ready

# 3. Chạy full bộ GPU integration tests
uv run pytest integration-tests -q

# 4. Cập nhật lại bộ bằng chứng 10/10 IP đạt 100 điểm
uv run lab28 evidence
```